# Arithmetic under two width budgets

Retain the complete three-input-bit domain and the original expression. Compile both max_qubits=10 and12 under objective=depth. Resource summaries replace graphical circuit display; seeded local CPU-probability samples replace cloud sampling. No promise of equal choices or resources to the source service. Run all cells from the repository root. See WORKFLOW.md for acceptance and adaptations.


In [ ]:
from typing import Annotated

import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import Output, QNum, allocate, hadamard_transform, max, qfunc


@qfunc
def main(
    x: Output[Annotated[QNum, 2]], y: Output[Annotated[QNum, 1]], z: Output[QNum]
) -> None:
    allocate(x)
    allocate(y)
    hadamard_transform(x)
    hadamard_transform(y)
    z |= (2 * x + y + max(3 * y, 2)) > 4



In [ ]:
from collections import Counter
from random import Random

programs = {}
for width in (10, 12):
    program = synth.synthesize(main, objective="depth", max_qubits=width)
    result = fq.run(program)
    probabilities = dict(program.probabilities(result, registers=("x", "y", "z")))
    counts = Counter(Random(17).choices(list(probabilities), weights=list(probabilities.values()), k=1000))
    programs[width] = program
    assert sum(counts.values()) == 1000
    print("Width budget:", width)
    print("Probabilities:", probabilities)
    print("Counts:", dict(counts))
    print("Resources:", program.report["resources"])
